In [0]:
CREATE OR REPLACE TEMPORARY VIEW silver_table_customer_olist AS
SELECT
customer_unique_id,
customer_zip_code_prefix,
customer_city,
customer_state,
silver_update_date
FROM ${catalog_olist}.${schema_silver}.${table_silver};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW silver_table_customer_olist_deduplicated AS
SELECT
*
FROM silver_table_customer_olist
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY silver_update_date DESC, customer_zip_code_prefix DESC) = 1;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW ${output_table} AS 
SELECT
XXHASH64(customer_unique_id) AS SK_CUSTOMER,
customer_unique_id AS ID_CUSTOMER,
CONCAT_WS(' - ', UPPER(customer_city), UPPER(customer_state)) AS CITY_STATE,
customer_zip_code_prefix AS ZIP_CODE
FROM silver_table_customer_olist_deduplicated;